# Assignment: Apply Q-Learning to the Lunar Lander Problem

Name: __________

SMU ID: __________


Learning Goals:
* Work with a discretized continuous problem
* Experiment with reward shaping

AI tool usage: 
* You **can use AI** to help you debug and write small pieces of code.

Instruction: Complete this notebook, run all cells, convert to HTML and upload to Canvas.

## Introduction

You want to use the TD control method Q-learning to learn a good policy for the Lunar Lander environment. The issue is that Lunar Lander has a continuous 
state space and the rewards (landing the vehicle) are extremely sparse.

## Setup

You need:
* Gymnasium (see [Installation Instructions](../common/Setup_Gymnasium.ipynb))
* Patched `gym-classics-1.0.0+internal.rev1` or later (see [Installation instructions](../common/Setup_patched_gym_classics.ipynb))

In [1]:
import numpy as np
np.set_printoptions(precision=2)

In [2]:
import gymnasium as gym
import gym_classics
gym_classics.register('gymnasium')

In [3]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py", 
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Reinforcement_Learning/refs/heads/main/common/")

## Discretizing the Observation Space

Lunar Lander has a continuous state space that needs to be discretized.
Gymnasium provides an `ObservationWrapper` that can be used to discretize observation on the fly. Here is how you use it:

In [4]:
from gymnasium.spaces import Box, MultiDiscrete
from gymnasium import ObservationWrapper

class ContinuousToDiscreteObs(ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = MultiDiscrete((10,10,10,10,10,10,2,2))

    def observation(self, obs):
        #  X = 0, Y = 1, VX = 2, VY = 3, ANGLE = 4, ANGULAR_VELOCITY = 5, LEFT_LEG_CONTACT = 6, RIGHT_LEG_CONTACT = 7
        # Orig. Observation Space: Box([ -2.5   -2.5  -10.   -10.    -6.28 -10.    -0.    -0.  ], [ 2.5   2.5  10.   10.    6.28 10.    1.    1.  ], (8,), float32)
        obs[0] = np.digitize(obs[0], bins = np.linspace(-2.5,2.5,10))
        obs[1] = np.digitize(obs[1], bins = np.linspace(-2.5,2.5,10))
        obs[2] = np.digitize(obs[2], bins = np.linspace(-10,10,10))
        obs[3] = np.digitize(obs[3], bins = np.linspace(-10,10,10))
        obs[4] = np.digitize(obs[4], bins = np.linspace(-6.28,6.28,10))
        obs[5] = np.digitize(obs[4], bins = np.linspace(-6.28,6.28,10))
        # Leg Contact is already discrete (obs[6], obs[7]) 
   
        return obs

Here is an example that show an episode with discretized observations.

In [5]:
from gymnasium_display_recorder import VideoWrapper, show

env = gym.make('LunarLander-v3', render_mode="rgb_array")
env = VideoWrapper(env, 'LL1', render_fps=30)
env = ContinuousToDiscreteObs(env)

obs, info = env.reset()
print (obs)

terminated = False
while not terminated:
    obs, reward, terminated, truncated, info = env.step(np.random.choice(range(3)))
    print (obs)

print(reward)

env.close()
show('LL1')    

/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5. 5. 9. 0. 0.]
[5. 8. 5. 5.

## Task 1: Experiment with Q-Learning

Apply Q-Learning to the problem.

Experiment with:
* How many episodes to use.
* What $\alpha$ and $\epsilon$ to use.
* Adapt the discretization to work better. Note: the discretization does not need to use equal spacing.

Add your final cleaned code below. You need to show at least:

* What is the success rate of your learned policy using a simulation with 100 tries.
* Create a video of one example landing. 

In [6]:
# Your final Code goes here.

Discuss what the major issues with learning in this example are.

`<Your discussion goes here>`

## Task 2: Experimenting with the Reward Signal

A significant issue is that the reward signal is very sparse, the agent gets no feedback till it lands or crashes. We can make 
the reward signal more dense by providing small rewards for getting closer to the goal. 
For example, there could be an additional reward for keeping the space crafts stable and centered.

Experiment with Q-learning while adding intermediate rewards using 
a [Reward Wrappers](https://gymnasium.farama.org/api/wrappers/reward_wrappers/). 

In [7]:
# Your final Code goes here.

Discuss what the major issues with learning in this example are.

`<Add your discussion>`

## Graduate Student Task

Reward shaping is a method to change the reward signal to make it easier for the agent to learn. A popular method is 
[potential-based reward shaping](https://ai.stanford.edu/~ang/papers/shaping-icml99.pdf).


Implement a simple strategy where the potential is defined by control and distance to the landing pad.

In [8]:
# Your final Code goes here.


Discuss how potential-based reward shaping can be used here. 

`<Add your discussion>`


&copy; 2025 [Michael Hahsler](http://michael.hahsler.net). 
This work is openly licensed under [Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0) License](https://creativecommons.org/licenses/by-sa/4.0/)

![CC BY-SA 4.0](https://licensebuttons.net/l/by-sa/3.0/88x31.png)